# 01 — Dataset exploration

This notebook walks through ECTSum end-to-end: loading from HF Hub, normalising columns, computing token-length distributions, and constructing the frozen 50-row evaluation hold-out.

Everything runs in `src/earningslora/data/` — this notebook only narrates and plots. If you find yourself rewriting logic here, lift it back into the package.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import matplotlib.pyplot as plt

from earningslora.config import get_settings
from earningslora.data.chunk import approx_token_count, fit_to_budget, split_sections
from earningslora.data.ectsum import load_ectsum, make_holdout, to_chat_format
from earningslora.data.stats import summary_length_stats, transcript_length_stats

settings = get_settings()
settings.model_dump()

## 1. Load the upstream dataset

`load_ectsum()` handles column-name variation across upstream mirrors and synthesises a `validation` split if one isn't provided. Cached locally by `datasets` after the first call.

In [ ]:
raw = load_ectsum()
for split, ds in raw.items():
    print(f"{split:12s}  rows={len(ds):>6}  cols={ds.column_names}")

In [ ]:
# Quick peek at one row
row = raw["train"][0]
print("--- transcript (first 600 chars) ---")
print(row["transcript"][:600])
print("\n--- summary ---")
print(row["summary"])

## 2. Length distributions

We're targeting a 4096-token training context. The ratio of transcripts that need truncation drives the section-aware chunker design in `data/chunk.py`.

In [ ]:
stats = {split: transcript_length_stats(list(ds)) for split, ds in raw.items()}
for split, s in stats.items():
    print(f"{split}: {s}")

In [ ]:
lengths = [approx_token_count(r["transcript"]) for r in raw["train"]]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lengths, bins=50)
ax.axvline(settings.max_seq_len, color="red", linestyle="--", label=f"max_seq_len={settings.max_seq_len}")
ax.set_xlabel("approx tokens (chars/4)")
ax.set_ylabel("transcripts")
ax.set_title("Train-split transcript length distribution")
ax.legend()
plt.show()

In [ ]:
summary_stats = {split: summary_length_stats(list(ds)) for split, ds in raw.items()}
for split, s in summary_stats.items():
    print(f"{split}: {s}")

## 3. Section detection

How clean is the prepared-remarks vs Q&A boundary? `split_sections` uses a regex over common header forms; let's count what it catches on a sample.

In [ ]:
sample = raw["train"].select(range(min(100, len(raw["train"]))))
n_qna_found = sum(1 for r in sample if split_sections(r["transcript"])["qna"])
n_boilerplate_found = sum(1 for r in sample if split_sections(r["transcript"])["boilerplate"])
print(f"Q&A header detected     : {n_qna_found}/{len(sample)}")
print(f"Boilerplate detected    : {n_boilerplate_found}/{len(sample)}")

## 4. Truncation impact

How many tokens do we lose to fit the 4096 budget?

In [ ]:
budget = settings.max_seq_len - 256  # leave headroom for the prompt + summary
before = []
after = []
for r in sample:
    before.append(approx_token_count(r["transcript"]))
    after.append(approx_token_count(fit_to_budget(r["transcript"], max_tokens=budget)))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist([before, after], bins=40, label=["before", "after"])
ax.axvline(budget, color="red", linestyle="--", label=f"budget={budget}")
ax.set_xlabel("approx tokens")
ax.set_ylabel("transcripts")
ax.legend()
ax.set_title("Transcript length: before vs after section-priority truncation")
plt.show()

## 5. Frozen hold-out

Same 50-row hold-out, deterministically carved with `seed=42`, used by every configuration in the bench.

In [ ]:
holdout = make_holdout(raw)
print(f"Hold-out size: {len(holdout)}")
print(f"First transcript (200 chars): {holdout[0]['transcript'][:200]}")

## 6. Chat-template formatting

Final step before training is wrapping each (transcript, summary) pair in the chat-template SFT shape — a `messages` list. Confirm one row round-trips correctly.

In [ ]:
chat = to_chat_format(raw)
print("Train cols:", chat["train"].column_names)
print("\nFirst record:")
print(json.dumps(chat["train"][0], indent=2)[:1500])

## Next

Notebook 02 runs the zero-shot baselines (base Llama 3.2 3B + Gemini 2.5 Flash) on this same hold-out, anchoring the metrics that the fine-tuned model in notebook 03 will try to beat.